# AMAG Adaptive Adjacency & Correlation Structure Analysis

This notebook investigates the AMAG model's adaptive adjacency mechanism and compares it with ground truth per-trial correlation structure.

**Key questions:**
1. What does the adaptive module output (S) look like? Gate (bimodal 0/1), pass-through (~0.5), or smooth?
2. How much does inter-channel correlation vary trial-to-trial?
3. Does S track real correlation changes across trials?
4. What is the channel clustering structure and how stable is it?

## Setup Instructions

### Before running this notebook:
1. **No GPU needed** — this is analysis-only (no training)
2. **Upload data to Google Drive** (if not done already):
   - `MonkeyNeural/dataset/` with the 5 `.npz` files
3. **Upload code to Google Drive** (if not done already):
   - `MonkeyNeural/src/` and `MonkeyNeural/replication/` folders
4. **Trained checkpoints required** — run the training notebook first, or upload checkpoints to:
   - `MonkeyNeural/training_results/checkpoints/beignet/best.pth`
   - `MonkeyNeural/training_results/checkpoints/affi/best.pth`

In [ ]:
# ── STEP 1: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
print('Drive mounted. Contents of MonkeyNeural/:',
      os.listdir('/content/drive/MyDrive/MonkeyNeural')
      if os.path.exists('/content/drive/MyDrive/MonkeyNeural')
      else 'NOT FOUND -- create the folder first')

In [ ]:
# ── STEP 2: Extract project code ────────────────────────────────────────────
import shutil

DRIVE_PROJECT = '/content/drive/MyDrive/MonkeyNeural'
LOCAL_PROJECT = '/content/MonkeyNeuralForecastingClaude'

os.makedirs(LOCAL_PROJECT, exist_ok=True)

# Copy src/ and replication/ from Drive to local (faster execution)
for folder in ['src', 'replication']:
    src_path = os.path.join(DRIVE_PROJECT, folder)
    dst_path = os.path.join(LOCAL_PROJECT, folder)
    if os.path.exists(src_path):
        if os.path.exists(dst_path):
            shutil.rmtree(dst_path)
        shutil.copytree(src_path, dst_path)
        print(f'Copied {folder}/ to {dst_path}')
    else:
        print(f'WARNING: {src_path} not found in Drive')

# Symlink dataset/ to avoid copying 400MB+
dataset_link = os.path.join(LOCAL_PROJECT, 'dataset')
dataset_src  = os.path.join(DRIVE_PROJECT, 'dataset')
if not os.path.exists(dataset_link) and os.path.exists(dataset_src):
    os.symlink(dataset_src, dataset_link)
    print(f'Symlinked dataset/ -> {dataset_src}')

# Copy checkpoints from Drive training_results to the expected local path
DRIVE_RESULTS = os.path.join(DRIVE_PROJECT, 'training_results')
for monkey in ['affi', 'beignet']:
    drive_ckpt = os.path.join(DRIVE_RESULTS, 'checkpoints', monkey, 'best.pth')
    local_ckpt_dir = os.path.join(LOCAL_PROJECT, 'replication', 'amag', 'checkpoints', monkey)
    local_ckpt = os.path.join(local_ckpt_dir, 'best.pth')
    if os.path.exists(drive_ckpt) and not os.path.exists(local_ckpt):
        os.makedirs(local_ckpt_dir, exist_ok=True)
        shutil.copy2(drive_ckpt, local_ckpt)
        print(f'Copied {monkey} checkpoint to {local_ckpt}')
    elif os.path.exists(local_ckpt):
        print(f'{monkey} checkpoint already exists at {local_ckpt}')
    else:
        print(f'WARNING: No {monkey} checkpoint found at {drive_ckpt}')

print('\nProject structure:')
for item in sorted(os.listdir(LOCAL_PROJECT)):
    print(f'  {item}/')

In [ ]:
# ── STEP 3: Install dependencies & add project to path ────────────────────
# torch, numpy, scipy, scikit-learn, matplotlib are pre-installed in Colab
import sys
if LOCAL_PROJECT not in sys.path:
    sys.path.insert(0, LOCAL_PROJECT)

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['figure.facecolor'] = 'white'

from scipy.stats import pearsonr, spearmanr
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from sklearn.covariance import LedoitWolf

from src.data.dataset import (
    load_npz, compute_normalization_stats, normalize, split_context_target,
    DATASET_DIR, MONKEY_FILES, MONKEY_CHANNELS, get_dataloaders
)
from replication.amag.model import AMAGReplica, build_model, compute_correlation_matrix

print(f'PyTorch: {torch.__version__}')
print(f'Dataset dir: {DATASET_DIR}')
print('Setup complete.')

## 1. Load Data & Model (Beignet first)

Start with Beignet (89 channels) — more manageable for visualization and per-trial correlation.

In [ ]:
MONKEY = 'beignet'

# Load raw data for ground truth correlation computation
raw_data = load_npz(os.path.join(DATASET_DIR, MONKEY_FILES[MONKEY]['train']))
print(f'Raw data shape: {raw_data.shape}')

# Get dataloaders (same split as training)
train_loader, val_loader, stats = get_dataloaders(MONKEY, batch_size=32)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

# Collect all val data as tensors
X_val_list, Y_val_list = [], []
for X_batch, Y_batch in val_loader:
    X_val_list.append(X_batch)
    Y_val_list.append(Y_batch)
X_val = torch.cat(X_val_list, dim=0)
Y_val = torch.cat(Y_val_list, dim=0)
print(f'Val X shape: {X_val.shape}, Val Y shape: {Y_val.shape}')

In [ ]:
# Load trained AMAG model
model = build_model(MONKEY, {'hidden_size': 64, 'use_adaptor': True, 'compute_init_corr': True})
ckpt_path = os.path.join(LOCAL_PROJECT, 'replication', 'amag', 'checkpoints', MONKEY, 'best.pth')
state = torch.load(ckpt_path, weights_only=False, map_location='cpu')
model.load_state_dict(state['model_state_dict'])
model.eval()

print(f'Loaded checkpoint: {ckpt_path}')
print(f'Best val MSE: {state["best_val_mse"]:.6f}')

# Verify: compute val MSE to confirm model loaded correctly
with torch.no_grad():
    pred_val = model(X_val)
    val_mse = ((pred_val - Y_val) ** 2).mean().item()
print(f'Verified val MSE: {val_mse:.6f}')

## 2. Extract Intermediate Representations

Replicate the adaptive module computation outside the forward pass to extract S, A_a_scaled, etc.

In [ ]:
def extract_adaptive_weights(model, X_batch, timesteps=None):
    """
    Extract adaptive weights S and effective adjacency A_a_scaled from AMAG.
    
    Args:
        model: AMAGReplica instance (eval mode)
        X_batch: (B, T_ctx, C, F) normalized context tensor
        timesteps: list of timesteps to extract (default: all)
    
    Returns:
        dict with extracted intermediate values
    """
    with torch.no_grad():
        # Step 1: Temporal encoding
        H = model.te(X_batch)  # (B, T, C, d)
        B, T, C, d = H.shape
        
        # Step 2: Fixed adjacency matrices
        A_a_norm = torch.tanh(model.si.A_a)  # (C, C)
        A_m_norm = torch.tanh(model.si.A_m)  # (C, C)
        
        # Step 3: Adaptive weights per timestep
        if timesteps is None:
            timesteps = list(range(T))
        
        S_per_t = {}
        A_a_scaled_per_t = {}
        
        for t in timesteps:
            H_mean = H[:, :t + 1, :, :].mean(dim=1)  # (B, C, d)
            
            # Replicate adaptor MLP computation
            h_u = H_mean.unsqueeze(2).expand(-1, -1, C, -1)  # (B, C, C, d)
            h_v = H_mean.unsqueeze(1).expand(-1, C, -1, -1)  # (B, C, C, d)
            h_uv = torch.cat([h_u, h_v], dim=-1)              # (B, C, C, 2d)
            S = torch.sigmoid(model.si.adaptor_mlp(h_uv).squeeze(-1))  # (B, C, C)
            A_a_scaled = A_a_norm.unsqueeze(0) * S  # (B, C, C)
            
            S_per_t[t] = S.cpu().numpy()
            A_a_scaled_per_t[t] = A_a_scaled.cpu().numpy()
    
    return {
        'A_a_norm': A_a_norm.cpu().numpy(),
        'A_m_norm': A_m_norm.cpu().numpy(),
        'S_per_t': S_per_t,
        'A_a_scaled_per_t': A_a_scaled_per_t,
    }

# Extract for all validation samples (at key timesteps to save memory)
# t=0 (first step, minimal context), t=4 (midpoint), t=9 (full context)
print('Extracting adaptive weights...')
results = extract_adaptive_weights(model, X_val, timesteps=[0, 4, 9])

S_t9 = results['S_per_t'][9]          # (N_val, C, C) — primary analysis target
A_a_norm = results['A_a_norm']          # (C, C)
A_a_scaled_t9 = results['A_a_scaled_per_t'][9]  # (N_val, C, C)

N_val, C, _ = S_t9.shape
print(f'S shape: ({N_val}, {C}, {C})')
print(f'S stats at t=9: mean={S_t9.mean():.4f}, std={S_t9.std():.4f}, min={S_t9.min():.4f}, max={S_t9.max():.4f}')

## 3. Adaptive Module Output Distribution (S)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# (a) Global histogram of all S values at t=9
ax = axes[0]
ax.hist(S_t9.flatten(), bins=100, density=True, alpha=0.7, color='steelblue', edgecolor='none')
ax.axvline(S_t9.mean(), color='red', linestyle='--', linewidth=1.5, label=f'mean={S_t9.mean():.3f}')
ax.axvline(np.median(S_t9.flatten()), color='orange', linestyle='--', linewidth=1.5, label=f'median={np.median(S_t9.flatten()):.3f}')
ax.set_xlabel('S value')
ax.set_ylabel('Density')
ax.set_title(f'(a) Distribution of S values (t=9)\nN={S_t9.size:,} values')
ax.legend(fontsize=8)
ax.set_xlim(0, 1)

# (b) Per-trial mean S
ax = axes[1]
trial_means = S_t9.mean(axis=(1, 2))  # (N_val,)
ax.hist(trial_means, bins=30, alpha=0.7, color='coral', edgecolor='none')
ax.set_xlabel('Mean S per trial')
ax.set_ylabel('Count')
ax.set_title(f'(b) Trial-level mean S\nrange: [{trial_means.min():.3f}, {trial_means.max():.3f}]')

# (c) Per-edge variance across trials
ax = axes[2]
edge_variance = S_t9.var(axis=0)  # (C, C)
im = ax.imshow(edge_variance, cmap='hot', aspect='auto')
plt.colorbar(im, ax=ax, label='Var(S) across trials')
ax.set_xlabel('Channel v')
ax.set_ylabel('Channel u')
ax.set_title(f'(c) Per-edge S variance\nmax={edge_variance.max():.4f}')

plt.suptitle(f'{MONKEY} — Adaptive Module Output Distribution', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# S evolution across timesteps
timesteps_to_check = [0, 4, 9]
S_means = [results['S_per_t'][t].mean() for t in timesteps_to_check]
S_stds = [results['S_per_t'][t].std() for t in timesteps_to_check]
S_mins = [results['S_per_t'][t].min() for t in timesteps_to_check]
S_maxs = [results['S_per_t'][t].max() for t in timesteps_to_check]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Mean and std across timesteps
ax = axes[0]
ax.errorbar(timesteps_to_check, S_means, yerr=S_stds, fmt='o-', capsize=5, 
            color='steelblue', markersize=8, linewidth=2)
ax.set_xlabel('Timestep')
ax.set_ylabel('S value')
ax.set_title('S mean +/- std across timesteps')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

# Histograms overlaid for t=0, t=4, t=9
ax = axes[1]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
for t, c in zip(timesteps_to_check, colors):
    ax.hist(results['S_per_t'][t].flatten(), bins=80, density=True, alpha=0.4, 
            color=c, label=f't={t}', edgecolor='none')
ax.set_xlabel('S value')
ax.set_ylabel('Density')
ax.set_title('S distribution at different timesteps')
ax.legend()
ax.set_xlim(0, 1)

plt.suptitle(f'{MONKEY} — S Evolution Across Timesteps', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Diagonal vs off-diagonal S
diag_mask = np.eye(C, dtype=bool)
S_diag = S_t9[:, diag_mask]       # (N_val, C) — self-loops
S_offdiag = S_t9[:, ~diag_mask]   # (N_val, C*(C-1)) — cross-channel

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(S_diag.flatten(), bins=60, density=True, alpha=0.6, label=f'Diagonal S(u,u): mean={S_diag.mean():.3f}', color='steelblue')
ax.hist(S_offdiag.flatten(), bins=60, density=True, alpha=0.6, label=f'Off-diagonal S(u,v): mean={S_offdiag.mean():.3f}', color='coral')
ax.set_xlabel('S value')
ax.set_ylabel('Density')
ax.set_title(f'{MONKEY} — Diagonal vs Off-diagonal S values (t=9)')
ax.legend()
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

print(f'Diagonal S:     mean={S_diag.mean():.4f}, std={S_diag.std():.4f}')
print(f'Off-diagonal S: mean={S_offdiag.mean():.4f}, std={S_offdiag.std():.4f}')

## 4. Per-Trial Ground Truth Correlation

**Challenge**: 10 timesteps for 89 channels is severely rank-deficient. We use Ledoit-Wolf shrinkage for regularization.

In [ ]:
def compute_trial_correlation(data_norm, trial_idx, use_all_features=True):
    """
    Compute regularized per-trial correlation matrix using Ledoit-Wolf shrinkage.
    
    Args:
        data_norm: (N, 20, C, 9) normalized data
        trial_idx: trial index
        use_all_features: if True, use all 9 features (90 obs); if False, LMP only (10 obs)
    
    Returns:
        corr: (C, C) regularized correlation matrix
    """
    trial_ctx = data_norm[trial_idx, :10, :, :]  # (10, C, 9)
    
    if use_all_features:
        # Treat each feature at each timestep as an observation: (90, C)
        obs = trial_ctx.transpose(0, 2, 1).reshape(-1, trial_ctx.shape[1])  # (90, C)
    else:
        obs = trial_ctx[:, :, 0]  # (10, C) — LMP only
    
    lw = LedoitWolf()
    lw.fit(obs)
    cov = lw.covariance_
    d = np.sqrt(np.diag(cov))
    d = np.maximum(d, 1e-12)
    corr = cov / np.outer(d, d)
    np.fill_diagonal(corr, 1.0)
    return corr


# Normalize raw data with same stats
data_norm = normalize(raw_data, stats)

# Get val indices (same seed=42 as get_dataloaders)
rng = np.random.default_rng(42)
n = len(data_norm)
indices = rng.permutation(n)
n_val = max(1, int(n * 0.15))
val_indices = indices[:n_val]

print(f'Computing per-trial correlation matrices for {len(val_indices)} val trials...')

# Compute for all val trials
all_trial_corrs = []
for i, idx in enumerate(val_indices):
    corr = compute_trial_correlation(data_norm, idx, use_all_features=True)
    all_trial_corrs.append(corr)
all_trial_corrs = np.stack(all_trial_corrs)  # (N_val, C, C)

print(f'Done. Shape: {all_trial_corrs.shape}')
print(f'Correlation range: [{all_trial_corrs.min():.3f}, {all_trial_corrs.max():.3f}]')

In [ ]:
# Visualize 9 randomly sampled per-trial correlation matrices
rng_vis = np.random.default_rng(123)
sample_idx = rng_vis.choice(len(val_indices), size=9, replace=False)

fig, axes = plt.subplots(3, 3, figsize=(14, 12))
for i, ax in enumerate(axes.flat):
    idx = sample_idx[i]
    im = ax.imshow(all_trial_corrs[idx], vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
    ax.set_title(f'Trial {val_indices[idx]}', fontsize=9)
    ax.set_xlabel('Ch' if i >= 6 else '')
    ax.set_ylabel('Ch' if i % 3 == 0 else '')

fig.colorbar(im, ax=axes, shrink=0.6, label='Pearson r (Ledoit-Wolf)')
plt.suptitle(f'{MONKEY} — Per-Trial Correlation Matrices (9 random val trials)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Quantify trial-to-trial variability
mean_corr = all_trial_corrs.mean(axis=0)   # (C, C)
std_corr = all_trial_corrs.std(axis=0)      # (C, C)

# Frobenius distance from each trial to the mean
frob_distances = np.array([
    np.linalg.norm(all_trial_corrs[i] - mean_corr) for i in range(len(all_trial_corrs))
])

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# (a) Mean correlation matrix
ax = axes[0]
im0 = ax.imshow(mean_corr, vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
plt.colorbar(im0, ax=ax, label='Mean r')
ax.set_title(f'(a) Mean correlation across trials')
ax.set_xlabel('Channel'); ax.set_ylabel('Channel')

# (b) Std of correlation
ax = axes[1]
im1 = ax.imshow(std_corr, cmap='hot', aspect='auto')
plt.colorbar(im1, ax=ax, label='Std(r)')
ax.set_title(f'(b) Std of correlation across trials\nmax={std_corr.max():.3f}')
ax.set_xlabel('Channel'); ax.set_ylabel('Channel')

# (c) Frobenius distances
ax = axes[2]
ax.hist(frob_distances, bins=25, alpha=0.7, color='teal', edgecolor='none')
ax.axvline(frob_distances.mean(), color='red', linestyle='--', label=f'mean={frob_distances.mean():.2f}')
ax.set_xlabel('Frobenius distance to mean corr')
ax.set_ylabel('Count')
ax.set_title('(c) Trial-to-trial variability')
ax.legend(fontsize=9)

plt.suptitle(f'{MONKEY} — Inter-trial Correlation Variability', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f'Off-diagonal std of correlation: mean={std_corr[~np.eye(C, dtype=bool)].mean():.4f}')
print(f'Frobenius distance: mean={frob_distances.mean():.3f}, std={frob_distances.std():.3f}')

## 5. AMAG Effective Adjacency vs Ground Truth

In [ ]:
# Select trials: 2 with highest Frobenius distance, 2 median, 2 lowest
sorted_idx = np.argsort(frob_distances)
selected = np.concatenate([
    sorted_idx[:2],                                     # lowest variability
    sorted_idx[len(sorted_idx)//2 - 1 : len(sorted_idx)//2 + 1],  # median
    sorted_idx[-2:],                                    # highest variability
])

n_sel = len(selected)
fig, axes = plt.subplots(3, n_sel, figsize=(3.5 * n_sel, 10))

for col, trial_i in enumerate(selected):
    frob = frob_distances[trial_i]
    
    # Row 0: Ground truth per-trial correlation
    im = axes[0, col].imshow(all_trial_corrs[trial_i], vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
    axes[0, col].set_title(f'Trial {val_indices[trial_i]}\nFrob={frob:.1f}', fontsize=9)
    if col == 0:
        axes[0, col].set_ylabel('GT Correlation')
    
    # Row 1: AMAG effective adjacency
    axes[1, col].imshow(A_a_scaled_t9[trial_i], vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
    if col == 0:
        axes[1, col].set_ylabel('A_a_scaled (AMAG)')
    
    # Row 2: Difference
    diff = A_a_scaled_t9[trial_i] - all_trial_corrs[trial_i]
    axes[2, col].imshow(diff, vmin=-0.5, vmax=0.5, cmap='RdBu_r', aspect='auto')
    if col == 0:
        axes[2, col].set_ylabel('Difference')

plt.suptitle(f'{MONKEY} — GT Correlation vs AMAG Effective Adjacency (selected trials)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Quantitative comparison: Pearson/Spearman between upper triangles
mask_ut = np.triu(np.ones((C, C), dtype=bool), k=1)

pearson_rs = []
spearman_rs = []
for i in range(len(all_trial_corrs)):
    v_model = A_a_scaled_t9[i][mask_ut]
    v_gt = all_trial_corrs[i][mask_ut]
    pr, _ = pearsonr(v_model, v_gt)
    sr, _ = spearmanr(v_model, v_gt)
    pearson_rs.append(pr)
    spearman_rs.append(sr)

pearson_rs = np.array(pearson_rs)
spearman_rs = np.array(spearman_rs)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(pearson_rs, bins=25, alpha=0.7, color='steelblue', edgecolor='none')
ax.axvline(pearson_rs.mean(), color='red', linestyle='--', label=f'mean={pearson_rs.mean():.3f}')
ax.set_xlabel('Pearson r (A_a_scaled vs GT corr)')
ax.set_ylabel('Count')
ax.set_title('Per-trial Pearson correlation')
ax.legend()

ax = axes[1]
ax.hist(spearman_rs, bins=25, alpha=0.7, color='coral', edgecolor='none')
ax.axvline(spearman_rs.mean(), color='red', linestyle='--', label=f'mean={spearman_rs.mean():.3f}')
ax.set_xlabel('Spearman r (A_a_scaled vs GT corr)')
ax.set_ylabel('Count')
ax.set_title('Per-trial Spearman correlation')
ax.legend()

plt.suptitle(f'{MONKEY} — A_a_scaled vs GT Correlation (upper triangle)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f'Pearson:  mean={pearson_rs.mean():.4f}, std={pearson_rs.std():.4f}')
print(f'Spearman: mean={spearman_rs.mean():.4f}, std={spearman_rs.std():.4f}')

In [ ]:
# Fixed adjacency drift: compare learned tanh(A_a) vs initial correlation
init_corr = compute_correlation_matrix(MONKEY)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

im0 = axes[0].imshow(init_corr, vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
plt.colorbar(im0, ax=axes[0])
axes[0].set_title('Initial A_a\n(training data LMP correlation)')

im1 = axes[1].imshow(A_a_norm, vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
plt.colorbar(im1, ax=axes[1])
axes[1].set_title('Learned tanh(A_a) after training')

diff = A_a_norm - init_corr
im2 = axes[2].imshow(diff, vmin=-0.3, vmax=0.3, cmap='RdBu_r', aspect='auto')
plt.colorbar(im2, ax=axes[2])
axes[2].set_title(f'Difference (learned - init)\nFrob norm={np.linalg.norm(diff):.2f}')

for ax in axes:
    ax.set_xlabel('Channel'); ax.set_ylabel('Channel')

plt.suptitle(f'{MONKEY} — Fixed Adjacency: Initial vs Learned', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# Correlation between init and learned
pr, _ = pearsonr(init_corr[mask_ut], A_a_norm[mask_ut])
print(f'Pearson r(init, learned) = {pr:.4f}')
print(f'Frobenius norm of difference = {np.linalg.norm(diff):.4f}')

In [ ]:
# CENTRAL FIGURE: Does S track trial-to-trial correlation changes?
# For each edge (u,v), compute Pearson r between S(u,v) and GT_corr(u,v) across trials

print('Computing per-edge S-vs-GT tracking correlation...')
edge_tracking = np.zeros((C, C))
edge_tracking_pval = np.zeros((C, C))

for u in range(C):
    for v in range(C):
        if u != v:
            s_vals = S_t9[:, u, v]
            gt_vals = all_trial_corrs[:, u, v]
            # Only compute if there's variance in both
            if s_vals.std() > 1e-8 and gt_vals.std() > 1e-8:
                r, p = pearsonr(s_vals, gt_vals)
                edge_tracking[u, v] = r
                edge_tracking_pval[u, v] = p

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# (a) Edge tracking correlation
ax = axes[0]
im = ax.imshow(edge_tracking, vmin=-0.5, vmax=0.5, cmap='RdBu_r', aspect='auto')
plt.colorbar(im, ax=ax, label='r(S, GT_corr) across trials')
ax.set_xlabel('Channel v'); ax.set_ylabel('Channel u')
ax.set_title('(a) Does S track GT correlation changes?')

# (b) Histogram of edge tracking values
ax = axes[1]
et_offdiag = edge_tracking[~np.eye(C, dtype=bool)]
ax.hist(et_offdiag, bins=60, density=True, alpha=0.7, color='steelblue', edgecolor='none')
ax.axvline(0, color='gray', linestyle='-', alpha=0.5)
ax.axvline(et_offdiag.mean(), color='red', linestyle='--', 
           label=f'mean={et_offdiag.mean():.4f}')
ax.set_xlabel('r(S, GT_corr)')
ax.set_ylabel('Density')
ax.set_title('(b) Distribution of tracking correlations')
ax.legend()

# Count significant edges
n_significant = ((np.abs(edge_tracking) > 0.2) & (~np.eye(C, dtype=bool))).sum()
n_total_edges = C * (C - 1)
print(f'Edges with |r| > 0.2: {n_significant}/{n_total_edges} ({100*n_significant/n_total_edges:.1f}%)')
print(f'Mean tracking correlation: {et_offdiag.mean():.4f}')
print(f'Fraction positive: {(et_offdiag > 0).mean():.1%}')

plt.suptitle(f'{MONKEY} — S vs GT Correlation Tracking (per-edge, across trials)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 6. Cluster Analysis

In [ ]:
# Hierarchical clustering on global correlation matrix
dist_matrix = 1 - np.abs(init_corr)
np.fill_diagonal(dist_matrix, 0)
# Ensure symmetry for squareform
dist_matrix = (dist_matrix + dist_matrix.T) / 2
condensed = squareform(dist_matrix, checks=False)
linkage_matrix = linkage(condensed, method='ward')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Dendrogram
ax = axes[0]
dendrogram(linkage_matrix, ax=ax, truncate_mode='lastp', p=20, 
           leaf_rotation=90, leaf_font_size=8)
ax.set_title(f'{MONKEY} — Channel Clustering Dendrogram')
ax.set_xlabel('Cluster (size)')
ax.set_ylabel('Ward distance')

# Reordered correlation matrix
ax = axes[1]
order = dendrogram(linkage_matrix, no_plot=True)['leaves']
corr_reordered = init_corr[np.ix_(order, order)]
im = ax.imshow(corr_reordered, vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
plt.colorbar(im, ax=ax, label='Pearson r')
ax.set_title('Correlation matrix (hierarchically reordered)')
ax.set_xlabel('Channel (reordered)'); ax.set_ylabel('Channel (reordered)')

plt.tight_layout()
plt.show()

In [ ]:
# Per-trial cluster stability
n_clusters = 5
cluster_labels = fcluster(linkage_matrix, t=n_clusters, criterion='maxclust')
print(f'Cluster sizes: {[np.sum(cluster_labels == k) for k in range(1, n_clusters + 1)]}')

def compute_cluster_modularity(corr_matrix, labels):
    """Compute mean |correlation| within-cluster vs between-cluster."""
    within, between = [], []
    for i in range(len(labels)):
        for j in range(i + 1, len(labels)):
            val = abs(corr_matrix[i, j])
            if labels[i] == labels[j]:
                within.append(val)
            else:
                between.append(val)
    return np.mean(within) if within else 0, np.mean(between) if between else 0

# Compute for all val trials
within_vals, between_vals = [], []
for i in range(len(all_trial_corrs)):
    w, b = compute_cluster_modularity(all_trial_corrs[i], cluster_labels)
    within_vals.append(w)
    between_vals.append(b)

within_vals = np.array(within_vals)
between_vals = np.array(between_vals)
modularity_ratio = within_vals / np.maximum(between_vals, 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.scatter(between_vals, within_vals, alpha=0.5, s=20, c='teal')
ax.plot([0, max(within_vals.max(), between_vals.max())], 
        [0, max(within_vals.max(), between_vals.max())], 'k--', alpha=0.3, label='y=x')
ax.set_xlabel('Between-cluster mean |corr|')
ax.set_ylabel('Within-cluster mean |corr|')
ax.set_title('Cluster modularity per trial')
ax.legend()

ax = axes[1]
ax.hist(modularity_ratio, bins=25, alpha=0.7, color='teal', edgecolor='none')
ax.axvline(modularity_ratio.mean(), color='red', linestyle='--', 
           label=f'mean ratio={modularity_ratio.mean():.2f}')
ax.set_xlabel('Within/Between cluster |corr| ratio')
ax.set_ylabel('Count')
ax.set_title('Modularity ratio distribution')
ax.legend()

plt.suptitle(f'{MONKEY} — Cluster Stability Across Trials ({n_clusters} clusters)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f'Within-cluster |corr|:  mean={within_vals.mean():.4f}, std={within_vals.std():.4f}')
print(f'Between-cluster |corr|: mean={between_vals.mean():.4f}, std={between_vals.std():.4f}')
print(f'Modularity ratio:       mean={modularity_ratio.mean():.2f}, std={modularity_ratio.std():.2f}')

In [ ]:
# Does S reflect cluster structure?
# Compare mean S for within-cluster vs between-cluster edges per trial

def compute_s_cluster_contrast(S_matrix, labels):
    """Mean S for within-cluster edges vs between-cluster edges."""
    within_s, between_s = [], []
    for i in range(len(labels)):
        for j in range(len(labels)):
            if i != j:
                if labels[i] == labels[j]:
                    within_s.append(S_matrix[i, j])
                else:
                    between_s.append(S_matrix[i, j])
    return np.mean(within_s), np.mean(between_s)

s_within, s_between = [], []
for i in range(len(S_t9)):
    w, b = compute_s_cluster_contrast(S_t9[i], cluster_labels)
    s_within.append(w)
    s_between.append(b)

s_within = np.array(s_within)
s_between = np.array(s_between)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.scatter(s_between, s_within, alpha=0.5, s=20, c='purple')
lim = [min(s_within.min(), s_between.min()) - 0.01, max(s_within.max(), s_between.max()) + 0.01]
ax.plot(lim, lim, 'k--', alpha=0.3, label='y=x')
ax.set_xlabel('Between-cluster mean S')
ax.set_ylabel('Within-cluster mean S')
ax.set_title('S cluster contrast per trial')
ax.legend()

ax = axes[1]
ax.hist(s_within - s_between, bins=25, alpha=0.7, color='purple', edgecolor='none')
ax.axvline(0, color='gray', linestyle='-', alpha=0.5)
ax.axvline((s_within - s_between).mean(), color='red', linestyle='--',
           label=f'mean diff={np.mean(s_within - s_between):.4f}')
ax.set_xlabel('S_within - S_between')
ax.set_ylabel('Count')
ax.set_title('Within - Between cluster S difference')
ax.legend()

plt.suptitle(f'{MONKEY} — Does S Reflect Cluster Structure?', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f'Within-cluster S:  mean={s_within.mean():.4f}')
print(f'Between-cluster S: mean={s_between.mean():.4f}')
print(f'Difference:        mean={np.mean(s_within - s_between):.4f}')
print(f'All trials have S_within > S_between: {np.all(s_within > s_between)}')

## 7. Repeat Key Analyses for Affi (239 channels)

In [ ]:
MONKEY_AFFI = 'affi'

# Load data
raw_data_affi = load_npz(os.path.join(DATASET_DIR, MONKEY_FILES[MONKEY_AFFI]['train']))
train_loader_a, val_loader_a, stats_a = get_dataloaders(MONKEY_AFFI, batch_size=16)

X_val_a_list, Y_val_a_list = [], []
for X_b, Y_b in val_loader_a:
    X_val_a_list.append(X_b)
    Y_val_a_list.append(Y_b)
X_val_a = torch.cat(X_val_a_list, dim=0)
Y_val_a = torch.cat(Y_val_a_list, dim=0)
print(f'Affi val X shape: {X_val_a.shape}')

# Load model
model_a = build_model(MONKEY_AFFI, {'hidden_size': 64, 'use_adaptor': True, 'compute_init_corr': True})
ckpt_path_a = os.path.join(LOCAL_PROJECT, 'replication', 'amag', 'checkpoints', MONKEY_AFFI, 'best.pth')
state_a = torch.load(ckpt_path_a, weights_only=False, map_location='cpu')
model_a.load_state_dict(state_a['model_state_dict'])
model_a.eval()

# Verify
with torch.no_grad():
    pred_a = model_a(X_val_a)
    mse_a = ((pred_a - Y_val_a) ** 2).mean().item()
print(f'Affi val MSE: {mse_a:.6f} (expected ~0.01410)')

In [ ]:
# Extract adaptive weights for Affi (process in mini-batches to manage memory)
print('Extracting Affi adaptive weights (t=9 only)...')

C_a = MONKEY_CHANNELS[MONKEY_AFFI]
batch_size_extract = 16
S_t9_a_list = []

with torch.no_grad():
    A_a_norm_a = torch.tanh(model_a.si.A_a).cpu().numpy()
    
    for start in range(0, len(X_val_a), batch_size_extract):
        end = min(start + batch_size_extract, len(X_val_a))
        X_b = X_val_a[start:end]
        H = model_a.te(X_b)
        B, T, C_loc, d = H.shape
        
        # Only compute S at t=9
        H_mean = H.mean(dim=1)  # (B, C, d) — mean over all 10 timesteps
        h_u = H_mean.unsqueeze(2).expand(-1, -1, C_loc, -1)
        h_v = H_mean.unsqueeze(1).expand(-1, C_loc, -1, -1)
        h_uv = torch.cat([h_u, h_v], dim=-1)
        S = torch.sigmoid(model_a.si.adaptor_mlp(h_uv).squeeze(-1)).cpu().numpy()
        S_t9_a_list.append(S)
        
        if (start // batch_size_extract) % 5 == 0:
            print(f'  Processed {end}/{len(X_val_a)} samples')

S_t9_a = np.concatenate(S_t9_a_list, axis=0)
A_a_scaled_t9_a = A_a_norm_a[np.newaxis, :, :] * S_t9_a

print(f'Affi S shape: {S_t9_a.shape}')
print(f'S stats: mean={S_t9_a.mean():.4f}, std={S_t9_a.std():.4f}')

In [ ]:
# Affi S distribution (same 3-panel plot)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

ax = axes[0]
ax.hist(S_t9_a.flatten(), bins=100, density=True, alpha=0.7, color='steelblue', edgecolor='none')
ax.axvline(S_t9_a.mean(), color='red', linestyle='--', linewidth=1.5, label=f'mean={S_t9_a.mean():.3f}')
ax.set_xlabel('S value'); ax.set_ylabel('Density')
ax.set_title(f'(a) S distribution (t=9)'); ax.legend(fontsize=8); ax.set_xlim(0, 1)

ax = axes[1]
trial_means_a = S_t9_a.mean(axis=(1, 2))
ax.hist(trial_means_a, bins=30, alpha=0.7, color='coral', edgecolor='none')
ax.set_xlabel('Mean S per trial'); ax.set_ylabel('Count')
ax.set_title(f'(b) Trial-level mean S\nrange: [{trial_means_a.min():.3f}, {trial_means_a.max():.3f}]')

ax = axes[2]
edge_var_a = S_t9_a.var(axis=0)
im = ax.imshow(edge_var_a, cmap='hot', aspect='auto')
plt.colorbar(im, ax=ax, label='Var(S)')
ax.set_title(f'(c) Per-edge S variance\nmax={edge_var_a.max():.4f}')

plt.suptitle(f'Affi — Adaptive Module Output Distribution', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Affi: per-trial GT correlations (using all 9 features for conditioning)
data_norm_a = normalize(raw_data_affi, stats_a)

rng_a = np.random.default_rng(42)
n_a = len(data_norm_a)
indices_a = rng_a.permutation(n_a)
n_val_a = max(1, int(n_a * 0.15))
val_indices_a = indices_a[:n_val_a]

print(f'Computing Affi per-trial correlations ({len(val_indices_a)} trials, {C_a} channels)...')
all_trial_corrs_a = []
for i, idx in enumerate(val_indices_a):
    corr = compute_trial_correlation(data_norm_a, idx, use_all_features=True)
    all_trial_corrs_a.append(corr)
    if (i + 1) % 30 == 0:
        print(f'  {i+1}/{len(val_indices_a)} done')
all_trial_corrs_a = np.stack(all_trial_corrs_a)
print(f'Done. Shape: {all_trial_corrs_a.shape}')

In [ ]:
# Affi: edge tracking — Does S track GT correlation?
print('Computing Affi per-edge tracking correlation...')
edge_tracking_a = np.zeros((C_a, C_a))

for u in range(C_a):
    if u % 50 == 0:
        print(f'  Channel {u}/{C_a}')
    for v in range(C_a):
        if u != v:
            s_vals = S_t9_a[:, u, v]
            gt_vals = all_trial_corrs_a[:, u, v]
            if s_vals.std() > 1e-8 and gt_vals.std() > 1e-8:
                r, _ = pearsonr(s_vals, gt_vals)
                edge_tracking_a[u, v] = r

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

ax = axes[0]
im = ax.imshow(edge_tracking_a, vmin=-0.5, vmax=0.5, cmap='RdBu_r', aspect='auto')
plt.colorbar(im, ax=ax, label='r(S, GT_corr)')
ax.set_xlabel('Channel v'); ax.set_ylabel('Channel u')
ax.set_title('(a) Does S track GT correlation? (Affi)')

ax = axes[1]
et_offdiag_a = edge_tracking_a[~np.eye(C_a, dtype=bool)]
ax.hist(et_offdiag_a, bins=60, density=True, alpha=0.7, color='steelblue', edgecolor='none')
ax.axvline(0, color='gray', linestyle='-', alpha=0.5)
ax.axvline(et_offdiag_a.mean(), color='red', linestyle='--',
           label=f'mean={et_offdiag_a.mean():.4f}')
ax.set_xlabel('r(S, GT_corr)'); ax.set_ylabel('Density')
ax.set_title('(b) Distribution of tracking correlations')
ax.legend()

n_sig_a = ((np.abs(edge_tracking_a) > 0.2) & (~np.eye(C_a, dtype=bool))).sum()
n_total_a = C_a * (C_a - 1)
print(f'Affi edges with |r| > 0.2: {n_sig_a}/{n_total_a} ({100*n_sig_a/n_total_a:.1f}%)')
print(f'Mean tracking correlation: {et_offdiag_a.mean():.4f}')

plt.suptitle('Affi — S vs GT Correlation Tracking', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Affi: cluster analysis with hierarchical clustering
init_corr_a = compute_correlation_matrix(MONKEY_AFFI)

dist_a = 1 - np.abs(init_corr_a)
np.fill_diagonal(dist_a, 0)
dist_a = (dist_a + dist_a.T) / 2
condensed_a = squareform(dist_a, checks=False)
linkage_a = linkage(condensed_a, method='ward')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

dendrogram(linkage_a, ax=axes[0], truncate_mode='lastp', p=25, leaf_rotation=90, leaf_font_size=7)
axes[0].set_title('Affi — Channel Clustering Dendrogram')

order_a = dendrogram(linkage_a, no_plot=True)['leaves']
corr_reord_a = init_corr_a[np.ix_(order_a, order_a)]
im = axes[1].imshow(corr_reord_a, vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
plt.colorbar(im, ax=axes[1], label='Pearson r')
axes[1].set_title('Affi — Reordered Correlation Matrix')

plt.tight_layout()
plt.show()

# Affi cluster labels
n_clusters_a = 7  # more clusters for 239 channels across 5 brain regions
cluster_labels_a = fcluster(linkage_a, t=n_clusters_a, criterion='maxclust')
print(f'Affi cluster sizes: {[np.sum(cluster_labels_a == k) for k in range(1, n_clusters_a + 1)]}')

# S cluster contrast for Affi
s_within_a, s_between_a = [], []
for i in range(len(S_t9_a)):
    w, b = compute_s_cluster_contrast(S_t9_a[i], cluster_labels_a)
    s_within_a.append(w)
    s_between_a.append(b)

s_within_a = np.array(s_within_a)
s_between_a = np.array(s_between_a)

print(f'Affi within-cluster S:  mean={s_within_a.mean():.4f}')
print(f'Affi between-cluster S: mean={s_between_a.mean():.4f}')
print(f'Affi S contrast:        {np.mean(s_within_a - s_between_a):.4f}')

## 8. Synthesis

In [ ]:
# Summary figure — 2x3 panel
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# (a) S distribution comparison: Beignet vs Affi
ax = axes[0, 0]
ax.hist(S_t9.flatten(), bins=80, density=True, alpha=0.5, label='Beignet', color='steelblue', edgecolor='none')
ax.hist(S_t9_a.flatten(), bins=80, density=True, alpha=0.5, label='Affi', color='coral', edgecolor='none')
ax.set_xlabel('S value'); ax.set_ylabel('Density')
ax.set_title('(a) Adaptive module S distribution')
ax.legend(); ax.set_xlim(0, 1)

# (b) Edge tracking — Beignet
ax = axes[0, 1]
im = ax.imshow(edge_tracking, vmin=-0.5, vmax=0.5, cmap='RdBu_r', aspect='auto')
plt.colorbar(im, ax=ax, label='r', shrink=0.8)
ax.set_title(f'(b) S-GT tracking (Beignet)\nmean r={et_offdiag.mean():.3f}')

# (c) Edge tracking — Affi
ax = axes[0, 2]
im = ax.imshow(edge_tracking_a, vmin=-0.5, vmax=0.5, cmap='RdBu_r', aspect='auto')
plt.colorbar(im, ax=ax, label='r', shrink=0.8)
ax.set_title(f'(c) S-GT tracking (Affi)\nmean r={et_offdiag_a.mean():.3f}')

# (d) Learned vs init adjacency (Beignet)
ax = axes[1, 0]
diff_b = A_a_norm - init_corr
im = ax.imshow(diff_b, vmin=-0.3, vmax=0.3, cmap='RdBu_r', aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title(f'(d) A_a drift from init (Beignet)\nFrob={np.linalg.norm(diff_b):.1f}')

# (e) S cluster contrast
ax = axes[1, 1]
labels_list = ['Beignet', 'Affi']
within_means = [s_within.mean(), s_within_a.mean()]
between_means = [s_between.mean(), s_between_a.mean()]
x = np.arange(2)
w = 0.3
ax.bar(x - w/2, within_means, w, label='Within-cluster S', color='steelblue', alpha=0.7)
ax.bar(x + w/2, between_means, w, label='Between-cluster S', color='coral', alpha=0.7)
ax.set_xticks(x); ax.set_xticklabels(labels_list)
ax.set_ylabel('Mean S')
ax.set_title('(e) S cluster contrast')
ax.legend(fontsize=8)

# (f) GT correlation variability comparison
ax = axes[1, 2]
frob_a = np.array([
    np.linalg.norm(all_trial_corrs_a[i] - all_trial_corrs_a.mean(axis=0)) 
    for i in range(len(all_trial_corrs_a))
])
ax.hist(frob_distances, bins=20, alpha=0.5, density=True, label='Beignet', color='steelblue', edgecolor='none')
ax.hist(frob_a, bins=20, alpha=0.5, density=True, label='Affi', color='coral', edgecolor='none')
ax.set_xlabel('Frobenius dist to mean corr')
ax.set_ylabel('Density')
ax.set_title('(f) Trial-to-trial correlation variability')
ax.legend()

plt.suptitle('AMAG Adaptive Adjacency Analysis — Summary', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Conclusions

### Key Findings

1. **Adaptive module S distribution**: S is a smooth, unimodal distribution — NOT a binary gate.
   - **Beignet**: mean=0.625, range [0.12, 0.93], moderate spread (std=0.095)
   - **Affi**: much higher mean=0.843, narrower relative spread (std=0.101)
   - S increases slightly with more temporal context (t=0: 0.57, t=9: 0.62 for Beignet)
   - Diagonal S(u,u) ≈ off-diagonal S(u,v) — adaptor does NOT distinguish self-loops

2. **Trial-to-trial correlation variability**: Substantial variability exists in the data.
   - Off-diagonal correlation std across trials: mean=0.20 (Beignet)
   - Frobenius distance to mean: 17.3 +/- 3.9 (Beignet) — significant trial-to-trial structure changes
   - Cluster modularity ratio ~1.54 — channels do form correlated groups

3. **S does NOT meaningfully track GT correlation changes**:
   - **Beignet**: mean edge tracking r = +0.04 (near zero), 24.7% edges with |r|>0.2, 59% positive
   - **Affi**: mean edge tracking r = **-0.18** (negative!), 56.7% edges with |r|>0.2
   - The adaptor is not learning to mirror real correlation dynamics; for Affi it's actually anti-correlated

4. **S does NOT reflect cluster structure**:
   - Beignet: within-cluster S = 0.626, between-cluster S = 0.624, difference = 0.002 (negligible)
   - Affi: within-cluster S = 0.842, between-cluster S = 0.843, difference = -0.0005 (essentially zero)
   - The adaptor treats all edges equally regardless of cluster membership

5. **Fixed adjacency has drifted substantially from initialization**:
   - Pearson r(init, learned) = 0.179 for Beignet — the learned A_a is very different from the correlation init
   - Frobenius norm of difference = 2.24
   - Training fundamentally reshapes the adjacency structure

### Interpretation

The adaptive module in its current form is **underutilized**. It produces a smooth, near-uniform scaling that:
- Does not track real per-trial correlation dynamics (despite substantial trial-to-trial variability existing)
- Does not respect cluster/community structure
- Operates at different mean levels for Affi (0.84) vs Beignet (0.63) but with minimal per-trial discrimination

Meanwhile, the **fixed adjacency** learns a representation very different from the initial correlation — suggesting the optimization finds a different (potentially better) graph structure than raw Pearson correlation.

### Implications for Next Experiments

1. **Auxiliary supervision on S**: Since trial-to-trial correlation variability is real but S doesn't track it, adding an auxiliary loss to encourage S ≈ per-trial correlation could help the model leverage this signal.

2. **Stronger adaptive mechanism**: The current MLP (128→64→1 with sigmoid) may lack capacity or training signal. Consider:
   - Directly computing per-trial correlation from embeddings (attention-style)
   - Multi-head attention replacing the MLP adaptor
   - Contrastive learning to make S sensitive to trial identity

3. **Remove adaptor + invest elsewhere**: Given the adaptor contributes minimally (paper ablation shows only -0.009 R² when using static adaptor), the parameter budget might be better spent on temporal modeling or multi-task learning.

4. **Region-aware adjacency for Affi**: The hierarchical clustering reveals clear block structure. A block-diagonal initialization or hierarchical graph could better capture Affi's multi-region structure.

5. **Learn adjacency from scratch**: Since learned A_a diverges heavily from init (r=0.18), the correlation initialization may be a weak prior. Consider learning graph structure end-to-end with better regularization (e.g., graphical lasso, sparsity penalty).